In [ ]:
# Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import math
import re

# Import sklearn libraries
from sklearn import svm
import sklearn.model_selection as ms
import sklearn.feature_extraction.text as text
import sklearn.naive_bayes as nb
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Data cleansing

In [ ]:
# Read in the statements
statements = pd.read_csv("fomc_statement.csv",parse_dates=["Date",])
yields = pd.read_csv("DailyTreasuryYield.csv",parse_dates=["Date",])
print(statements.head())
print(yields.head())
print(len(statements))
print(len(yields))

In [ ]:
# Correct one FOMC website mistake
statements.loc[statements["Date"] == datetime.date(2007, 6, 18), "Date"] = datetime.date(2007, 6, 28)
statements.sort_values(by='Date', inplace=True)
# Change a weekend to the latest business date
statements.loc[statements["Date"] == datetime.date(2010, 5, 9), "Date"] = datetime.date(2010, 5, 7)

In [ ]:
plt.plot(yields["Date"],yields["10 YR"] - yields["3 MO"])

In [ ]:
result = pd.merge(yields, statements, on='Date', how='left')
result['10YR_3MO'] = result['10 YR']-result['3 MO']
result['d3MO'] = result['3 MO'].diff()
result['d_structure'] = result['10YR_3MO'].diff()

In [ ]:
result.head(10)

In [ ]:
result[result['Statement'].notnull()].head(10)

In [ ]:
# generate rates movement for each release date
myDates = result[result['Statement'].notnull()]['Date']
window = 3 # business days
for i in range(len(myDates)):
    start_index = myDates.index[i]
    if i == len(myDates) - 1:
        policy_interim = np.inf
    else: 
        policy_interim = myDates.index[i+1] - myDates.index[i]
    sub_result = result.iloc[start_index:min(start_index+window,start_index+policy_interim)]
    result.loc[result["Date"] == myDates.iloc[i], "3MO_move"] = sub_result['d3MO'].sum()
    result.loc[result["Date"] == myDates.iloc[i], "3MO_std"] = sub_result['d3MO'].std()
    result.loc[result["Date"] == myDates.iloc[i], "structure_move"] = sub_result['d_structure'].sum()
    result.loc[result["Date"] == myDates.iloc[i], "structure_std"] = sub_result['d_structure'].std()

In [ ]:
out = result[result['Statement'].notnull()]
out.head(5)

In [ ]:
plt.plot(out['3MO_move'], out['3MO_std'],'*')

In [ ]:
plt.plot(out['structure_move'], np.log(out['structure_std']),'.')

In [ ]:
out['3MO_label'] = out['3MO_move'] > 0
out['structure_label'] = out['structure_move'] > 0

In [ ]:
out.head()

# Model

In [ ]:
# Identify labels
y = out['structure_label'] #out['3MO_label']

# Implement Tfid vectorizer
vectorizer = text.TfidfVectorizer(stop_words=['about','the','committee', "ben", "geithner", "bernanke", "timothy", "hoenig", "thomas", "donald", "kevin", "mishkin", "kroszner", "kohn", "charles", "frederic"])

def textfilter(x):
    out = re.sub(r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)', '',x)
    out = re.sub(r'\d+','',x)
    return out

X = vectorizer.fit_transform(map(textfilter,out['Statement'].values))
print(X.shape)

In [ ]:
print(vectorizer.get_feature_names())

In [ ]:
print(vectorizer.get_stop_words())

In [ ]:
seed = 10
X_train, X_test, y_train, y_test = ms.train_test_split(X, y, test_size=0.1,random_state=seed)

In [ ]:
y.describe()

## Prediction

### Naive Bayes

In [ ]:
# Use GridSearchCV to find optimal alpha
nb_model = ms.GridSearchCV(nb.BernoulliNB(), param_grid={'alpha': np.logspace(-2., 2., 50)})
nb_model.fit(X_train, y_train)

In [ ]:
nb_model.score(X_test, y_test)

### SVM

In [ ]:
svm_model = ms.GridSearchCV(svm.SVC(kernel='rbf'), 
                            {'C': [0.001, 0.01, 0.1, 1, 10], 'gamma' : [0.001, 0.01, 0.1, 1]}, 
                            cv=5)
svm_model.fit(X_train, y_train)

In [ ]:
svm_model.score(X_test, y_test)

In [ ]:
y_test.describe()

## Sentiment Analysis

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
stop_words = list(set(stopwords.words('english')))
type(stop_words)

In [ ]:
import pysentiment as ps

In [ ]:
def assignSentmentScore(x):
    lm = ps.LM()
    tokens = lm.tokenize(x)
    score = lm.get_score(tokens)
    return score['Polarity']

In [ ]:
sentimentScore = list(map(assignSentmentScore,out['Statement'].values))

In [ ]:
len(sentimentScore)

In [ ]:
plt.plot(out['Date'],sentimentScore)